# Elderly Care Risk Prediction
## Exploratory Data Analysis & Model Building
**Dataset:** LASI Wave 1 (Longitudinal Ageing Study in India)  
**Author:** Tushar Sekhri  
**Date:** July 2026

In [15]:
import pyreadstat
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("All imports successful")

All imports successful


In [16]:
#Load the main LASI dataset
df,meta=pyreadstat.read_dta('../data/3_LASI_W1_Individual_v4.dta')

print(f"Dataset shape: {df.shape}")
print(f"Rows: {df.shape[0]:,} individuals")
print(f"Columns: {df.shape[1]:,} variables")

Dataset shape: (73396, 2623)
Rows: 73,396 individuals
Columns: 2,623 variables


In [17]:
#import sys
#!{sys.executable} -m pip install pyreadstat
#import sys
#!{sys.executable} -m pip install pyreadstat pandas numpy matplotlib seaborn scikit-learn

In [18]:
# Selecting required features
features = [
    'dm003', 'dm005',           # gender, age
    'ht001_a',                  # self rated health
    'ht002', 'ht003', 'ht004', 
    'ht005', 'ht006', 'ht007', 
    'ht008', 'ht009', 'ht010',  # diseases
    'ht301',                    # physical impairment
    'ht401', 'ht402', 'ht403', 
    'ht404', 'ht405', 'ht406',  # ADL
    'ht407', 'ht408', 'ht409', 
    'ht410', 'ht411', 'ht412',  # IADL
    'mh101', 'mh104', 'mh105',  # memory
    'living_arrangements',      # lives alone or not
]

ml_df=df[features].copy()


# Create target variable
# In LASI: 1 = has difficulty, 2 = no difficulty
adl_cols = ['ht401','ht402','ht403','ht404','ht405','ht406',
            'ht407','ht408','ht409','ht410','ht411','ht412']


ml_df['needs_care'] = (ml_df[adl_cols] == 1).any(axis=1).astype(int)

print("Target variable distribution:")
print(ml_df['needs_care'].value_counts())
print(f"\n{ml_df['needs_care'].mean()*100:.1f}% of people need care support")


Target variable distribution:
needs_care
0    49174
1    24222
Name: count, dtype: int64

33.0% of people need care support


In [19]:
#checking missing values per column
print("Missing values per columns:",ml_df.isnull().sum().sort_values(ascending=False))



Missing values per columns: mh104                  72681
mh101                  72681
mh105                  72681
ht001_a                  947
ht409                    367
ht301                    325
ht408                    314
ht404                    309
ht412                    309
ht411                    309
ht410                    309
ht407                    309
ht406                    309
ht405                    309
ht403                    309
ht402                    309
ht401                    309
ht003                    196
ht009                    195
ht002                    193
ht010                    188
ht004                    188
ht007                    187
ht006                    186
ht005                    186
ht008                    185
living_arrangements        0
dm003                      0
dm005                      0
needs_care                 0
dtype: int64


In [20]:
# Drop memory columns - 99% missing, unusable
ml_df_clean = ml_df.drop(columns=['mh101', 'mh104', 'mh105'])

# Updated feature list without memory columns
adl_cols = ['ht401','ht402','ht403','ht404','ht405','ht406',
            'ht407','ht408','ht409','ht410','ht411','ht412']

# Drop rows where ADL columns are missing (only 309 rows - very few)
ml_df_clean=ml_df_clean.dropna(subset=adl_cols)

#fill others by mean
ml_df_clean=ml_df_clean.fillna(ml_df_clean.median(numeric_only=True))

print(f"Clean dataset shape{ml_df_clean.shape}")
print(f"dropped rows : {len(ml_df)-len(ml_df_clean):,}")
print(f"missing values now: {ml_df_clean.isnull().sum().sum()}")
print(f"\nTarget distribution:")
print(ml_df_clean['needs_care'].value_counts())

Clean dataset shape(73027, 27)
dropped rows : 369
missing values now: 881

Target distribution:
needs_care
0    48816
1    24211
Name: count, dtype: int64


In [21]:
# Check which columns still have missing values
print("Columns still with missing values:")
print(ml_df_clean.isnull().sum()[ml_df_clean.isnull().sum() > 0])

Columns still with missing values:
ht001_a    726
ht002       18
ht003       21
ht004       13
ht005       11
ht006       11
ht007       12
ht008       10
ht009       19
ht010       13
ht301       27
dtype: int64


In [22]:
# Fill remaining missing values with mode (most frequent value)
# These are categorical columns (1=yes, 2=no) so mode makes more sense than median
cols_with_missing = ['ht001_a', 'ht002', 'ht003', 'ht004', 'ht005', 
                     'ht006', 'ht007', 'ht008', 'ht009', 'ht010', 'ht301']

for col in cols_with_missing:
    mode_val = ml_df_clean[col].mode()[0]
    ml_df_clean[col] = ml_df_clean[col].fillna(mode_val)

print(f"Missing values remaining: {ml_df_clean.isnull().sum().sum()}")
print(f"Dataset shape: {ml_df_clean.shape}")

Missing values remaining: 0
Dataset shape: (73027, 27)
